# EEG Sleep Stage Classification with a 1D CNN
**Author:** Andres Quintanar  
**Dataset:** [OpenNeuro ds005555](https://openneuro.org/datasets/ds005555) — 128-subject overnight PSG + wearable headband EEG  
**Framework:** PyTorch  
**Environment:** Google Colab (GPU recommended)

---

## Abstract
Sleep disorders affect millions of people worldwide, yet diagnosis relies on costly, clinic-based
polysomnography (PSG) that requires expert manual interpretation. Electroencephalography (EEG)
non-invasively captures electrical brain activity and can distinguish the distinct neural signatures
of each sleep stage, making it a strong candidate for automated, wearable-based sleep monitoring.

This project developed a machine learning model to classify single-channel EEG recordings into
five sleep stages: Wake, N1, N2, N3, and REM.

Continuous overnight recordings were segmented
into 30-second epochs aligned to expert PSG annotations.

A three-block 1D Convolutional Neural
Network (CNN) was trained on these epochs using PyTorch, with batch normalization, max pooling,
dropout regularization, and inverse-frequency class weighting to handle severe class imbalance.

The 5-class model achieved 66.82 % validation accuracy; a 3-class model (merging N1/N2/N3
into a single NREM class) reached 82.2%, demonstrating that a lightweight CNN trained on a
single wearable EEG channel can reliably distinguish sleep stages.

---

## Notebook Overview
| Step | Section | Description |
|------|---------|-------------|
| 1 | [Install & Import](#1-install-and-import-dependencies) | Install packages, import libraries |
| 2 | [Load Raw Data](#2-load-raw-eeg-data) | Download and sanity-check one subject |
| 3 | [Preprocess All Subjects](#3-preprocess-and-save-all-subjects) | Extract epochs, save to Drive |
| 4 | [Build DataLoaders](#4-build-memory-safe-dataloaders) | Memory-safe iterable dataset |
| 5 | [Split & Normalize](#5-train--validation-split-and-normalization) | Subject-level split + z-score norm |
| 6 | [EDA – Label Distribution](#6-exploratory-data-analysis) | Count epochs per sleep stage |
| 7 | [5-Class CNN](#7-model-architecture--5-class-cnn) | Architecture definition |
| 8 | [Train 5-Class Model](#8-train-the-5-class-model) | Full training loop |
| 9 | [Results – 5-Class](#9-results--5-class-model) | Accuracy curves, confusion matrix |
| 10 | [3-Class Model](#10-3-class-model-wake--nrem--rem) | Collapsed NREM variant |


## 1. Install and Import Dependencies
> Run this cell first. It installs all required packages into the Colab runtime.


In [ ]:
# Install
!pip install -q mne numpy pandas scikit-learn torch awscli

# Standard library
import os
import gc
import glob
import random

# Numerical / data
import numpy as np
import pandas as pd

# EEG
import mne
mne.set_log_level("WARNING")   # suppress verbose MNE output

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader

# Evaluation / visualization
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

print(f"PyTorch  : {torch.__version__}")
print(f"MNE      : {mne.__version__}")
print(f"Device   : {'CUDA' if torch.cuda.is_available() else 'CPU'}")


## 2. Load Raw EEG Data
Downloads the headband EEG `.edf` files and matching PSG event `.tsv` files directly
from the OpenNeuro AWS S3 bucket. The cell then performs a quick sanity check by
loading the first matched subject pair into MNE Epochs.

> **Note:** The full sync (~128 subjects) takes several minutes.
> The sanity-check block loads only the **first** verified subject pair.


In [ ]:
# 1. Create dataset directory and download
os.makedirs('/content/ds005555', exist_ok=True)
os.chdir('/content/ds005555')

print("Downloading headband EEG and PSG event files from OpenNeuro S3...")
!aws s3 sync --no-sign-request s3://openneuro.org/ds005555 . \
    --exclude "*" \
    --include "sub-*/eeg/*_task-Sleep_acq-headband_eeg.edf" \
    --include "sub-*/eeg/*_task-Sleep_acq-psg_events.tsv"

# 2. Match EEG files with their event files
eeg_files     = sorted(glob.glob('sub-*/eeg/*_task-Sleep_acq-headband_eeg.edf'))
verified_pairs = []

for eeg_path in eeg_files:
    event_path = eeg_path.replace('acq-headband_eeg.edf', 'acq-psg_events.tsv')
    if os.path.exists(event_path):
        verified_pairs.append((eeg_path, event_path))
    else:
        print(f"  Warning: missing event file for {eeg_path}")

print(f"\nMatched {len(verified_pairs)} subject pairs.")

# 3. Sanity check: load the first subject
if verified_pairs:
    eeg_path, events_path = verified_pairs[0]
    print(f"\nLoading: {eeg_path}")

    raw       = mne.io.read_raw_edf(eeg_path, preload=True, verbose=False)
    events_df = pd.read_csv(events_path, sep='\t')

    annotations = mne.Annotations(
        onset       = events_df['onset'].values,
        duration    = events_df['duration'].values,
        description = events_df['stage_hum'].astype(str).values,
    )
    raw.set_annotations(annotations)
    events, event_id = mne.events_from_annotations(raw, verbose=False)

    epochs = mne.Epochs(
        raw, events, event_id,
        tmin=0, tmax=30, baseline=None, preload=True, verbose=False,
    )
    print("\nEpochs created successfully:")
    print(epochs)
else:
    print("No valid pairs found — check the download step above.")


## 3. Preprocess and Save All Subjects
Iterates over every verified subject pair, extracts 30-second EEG epochs,
and saves the resulting tensors (`X` = signal, `y` = stage label) to Google Drive.

**Skip logic:** subjects whose output files already exist are skipped, so the cell
is safe to re-run after an interrupted session.

> ⏱ Expect ~2–3 hours for the full 128-subject dataset.


In [ ]:
from google.colab import drive

# Mount Drive and configure paths
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/Sleep Classification Project/Processed EEG Data"
os.makedirs(SAVE_DIR, exist_ok=True)

# Sleep-stage label mapping (artifact epochs are labelled 5–8 in the TSV)
SLEEP_STAGE_MAP = {
    '0': 0,   # Wake
    '1': 1,   # N1
    '2': 2,   # N2
    '3': 3,   # N3
    '4': 4,   # REM
}

# Re-build verified_pairs if running this cell standalone
if 'verified_pairs' not in dir() or not verified_pairs:
    os.chdir('/content/ds005555')
    eeg_files      = sorted(glob.glob('sub-*/eeg/*_task-Sleep_acq-headband_eeg.edf'))
    verified_pairs = [
        (p, p.replace('acq-headband_eeg.edf', 'acq-psg_events.tsv'))
        for p in eeg_files
        if os.path.exists(p.replace('acq-headband_eeg.edf', 'acq-psg_events.tsv'))
    ]

total = len(verified_pairs)
print(f"Processing {total} subjects -> {SAVE_DIR}\n")

for i, (eeg_path, events_path) in enumerate(verified_pairs, 1):
    sub_id   = eeg_path.split('/')[0]
    x_out    = os.path.join(SAVE_DIR, f"{sub_id}_X.pt")
    y_out    = os.path.join(SAVE_DIR, f"{sub_id}_y.pt")

    if os.path.exists(x_out) and os.path.exists(y_out):
        print(f"  [{i:3d}/{total}] {sub_id} — already processed, skipping.")
        continue

    try:
        print(f"  [{i:3d}/{total}] {sub_id} ...", end=" ", flush=True)

        raw       = mne.io.read_raw_edf(eeg_path, preload=True, verbose=False)
        events_df = pd.read_csv(events_path, sep='\t')

        # Keep only valid sleep-stage annotations
        valid = events_df[events_df['stage_hum'].isin([0, 1, 2, 3, 4])]
        annotations = mne.Annotations(
            onset       = valid['onset'].values,
            duration    = valid['duration'].values,
            description = valid['stage_hum'].astype(str).values,
        )
        raw.set_annotations(annotations)

        events, _ = mne.events_from_annotations(
            raw, event_id=SLEEP_STAGE_MAP, verbose=False,
        )
        epochs = mne.Epochs(
            raw, events, event_id=SLEEP_STAGE_MAP,
            tmin=0, tmax=30, baseline=None, preload=True, verbose=False,
        )

        X_tensor = torch.tensor(epochs.get_data(),    dtype=torch.float32)
        y_tensor = torch.tensor(epochs.events[:, 2],  dtype=torch.long)

        torch.save(X_tensor, x_out)
        torch.save(y_tensor, y_out)
        print(f"saved ({len(y_tensor)} epochs).")

    except Exception as exc:
        print(f"FAILED — {exc}")

    finally:
        # Free RAM before the next subject
        for var in ['raw', 'epochs', 'X_tensor', 'y_tensor',
                    'events_df', 'valid', 'annotations', 'events']:
            if var in dir():
                del locals()[var]
        gc.collect()

print("\n--- Preprocessing complete ---")


## 4. Build Memory-Safe DataLoaders
Loading all ~128 subjects into RAM at once would exceed Colab's memory limit.
`SleepIterableDataset` streams one subject at a time: it loads, yields all
epochs for that subject, then immediately deletes the tensors before moving on.


In [ ]:
class SleepIterableDataset(IterableDataset):
    """
    Streams pre-processed EEG tensors from disk one subject at a time.

    Parameters
    ----------
    x_paths : list[str]
        Paths to *_X.pt files (shape: [n_epochs, 1, n_timepoints]).
    y_paths : list[str]
        Paths to matching *_y.pt label files.
    shuffle : bool
        If True, shuffle subject order AND epoch order within each subject.
    label_map : dict | None
        Optional integer-to-integer remapping applied to every label
        (e.g. {0:0, 1:1, 2:1, 3:1, 4:2} for the 3-class collapse).
    """

    VALID_LABELS = {0, 1, 2, 3, 4}   # artifact epochs have label >= 5

    def __init__(self, x_paths, y_paths, shuffle=True, label_map=None):
        assert len(x_paths) == len(y_paths), "Mismatch between X and y file lists."
        self.x_paths   = x_paths
        self.y_paths   = y_paths
        self.shuffle   = shuffle
        self.label_map = label_map  # None → identity (5-class mode)

    def __iter__(self):
        indices = list(range(len(self.x_paths)))
        if self.shuffle:
            np.random.shuffle(indices)

        for idx in indices:
            x_path, y_path = self.x_paths[idx], self.y_paths[idx]
            if not (os.path.exists(x_path) and os.path.exists(y_path)):
                continue

            X_sub = torch.load(x_path, weights_only=True)
            y_sub = torch.load(y_path, weights_only=True)

            ep_idx = list(range(len(y_sub)))
            if self.shuffle:
                np.random.shuffle(ep_idx)

            for ei in ep_idx:
                orig = int(y_sub[ei].item())
                if orig not in self.VALID_LABELS:
                    continue

                label   = self.label_map[orig] if self.label_map else orig
                X_epoch = X_sub[ei, 0:1, :]      # single headband channel

                # Per-epoch z-score normalization
                std = X_epoch.std()
                X_epoch = (X_epoch - X_epoch.mean()) / (std if std > 1e-6 else 1.0)

                yield X_epoch, torch.tensor(label, dtype=torch.long)

            del X_sub, y_sub, ep_idx
            gc.collect()


## 5. Train / Validation Split and Normalization
The split is performed **at the subject level** (not epoch level) to prevent
data leakage: no subject appears in both train and validation sets.


In [ ]:
# Configuration
SEED      = 42
VAL_SPLIT = 0.15
DATA_DIR  = "/content/drive/MyDrive/Sleep Classification Project/Processed EEG Data"
BATCH_SIZE = 64

# Collect all processed files
all_x = sorted(glob.glob(os.path.join(DATA_DIR, '*_X.pt')))
all_y = sorted(glob.glob(os.path.join(DATA_DIR, '*_y.pt')))

if not all_x:
    raise FileNotFoundError(
        f"No processed files found in {DATA_DIR}.\n"
        "Please run Section 3 (Preprocess All Subjects) first."
    )

# Subject-level random split
random.seed(SEED)
val_size = max(1, int(len(all_x) * VAL_SPLIT))
val_ids  = set(random.sample(range(len(all_x)), val_size))

train_x = [p for i, p in enumerate(all_x) if i not in val_ids]
train_y = [p for i, p in enumerate(all_y) if i not in val_ids]
val_x   = [p for i, p in enumerate(all_x) if i     in val_ids]
val_y   = [p for i, p in enumerate(all_y) if i     in val_ids]

print(f"Train subjects : {len(train_x)}")
print(f"Val   subjects : {len(val_x)}")

# Build 5-class loaders
train_dataset = SleepIterableDataset(train_x, train_y, shuffle=True)
val_dataset   = SleepIterableDataset(val_x,   val_y,   shuffle=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

# Sanity check
X_b, y_b = next(iter(train_loader))
print(f"\nSanity check — batch X : {X_b.shape}")
print(f"               batch y : {y_b.shape}")
print(f"               labels  : {torch.unique(y_b).tolist()}")
print(f"               mean≈0  : {X_b[0].mean().item():.4f}")
print(f"               std≈1   : {X_b[0].std().item():.4f}")


## 6. Exploratory Data Analysis
Count the number of labeled epochs per sleep stage across all subjects.
This reveals the class imbalance that motivates inverse-frequency weighting during training.


In [ ]:
STAGE_NAMES = ["Wake", "N1", "N2", "N3", "REM"]

y_paths      = sorted(glob.glob(os.path.join(DATA_DIR, '*_y.pt')))
total_counts = {i: 0 for i in range(5)}

print(f"Scanning {len(y_paths)} subjects...")
for yp in y_paths:
    y = torch.load(yp, weights_only=True)
    for label, count in zip(*torch.unique(y, return_counts=True)):
        label = int(label.item())
        if label in total_counts:
            total_counts[label] += int(count.item())

print("\n--- Label Distribution ---")
total_epochs = sum(total_counts.values())
for label, count in sorted(total_counts.items()):
    pct = 100 * count / total_epochs
    print(f"  {STAGE_NAMES[label]:5s} (label {label}): {count:6d} epochs  ({pct:.1f} %)")

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
colors  = ["steelblue", "salmon", "mediumseagreen", "mediumpurple", "goldenrod"]
bars    = ax.bar(STAGE_NAMES, [total_counts[i] for i in range(5)], color=colors)
ax.set_ylabel("Epoch Count", fontsize=11)
ax.set_title("Class Distribution Across All Subjects", fontsize=13)
for bar, (_, cnt) in zip(bars, sorted(total_counts.items())):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 200,
        f"{cnt:,}", ha="center", fontsize=9,
    )
fig.tight_layout()
plt.show()


## 7. Model Architecture — 5-Class CNN

A lightweight **three-block 1D CNN** designed for wearable EEG:

| Block | Layer | Filters | Kernel | Stride | Padding | Follow-up |
|-------|-------|---------|--------|--------|---------|-----------|
| 1 | Conv1d | 16 | 31 | 2 | 15 | BN → ReLU → MaxPool(4) |
| 2 | Conv1d | 32 | 15 | 2 | 7  | BN → ReLU → MaxPool(4) |
| 3 | Conv1d | 64 |  7 | 1 | 3  | BN → ReLU → MaxPool(2) |
| — | AdaptiveAvgPool1d(16) | — | — | — | — | flattens to 1024 |
| — | FC(1024 → 128) → Dropout(0.5) → FC(128 → 5) | — | — | — | — | output logits |

Large kernels in early layers capture slow delta/theta rhythms; smaller kernels in later layers
detect faster spindle-like oscillations.


In [ ]:
class SleepCNN(nn.Module):
    """
    Three-block 1D CNN for single-channel EEG sleep stage classification.

    Parameters
    ----------
    num_classes : int
        Number of output classes (5 for full staging, 3 for Wake/NREM/REM).
    """

    def __init__(self, num_classes: int = 5):
        super().__init__()

        # Block 1 — captures slow rhythms (~0.5–4 Hz range)
        self.conv1 = nn.Conv1d(1, 16, kernel_size=31, stride=2, padding=15)
        self.bn1   = nn.BatchNorm1d(16)
        self.pool1 = nn.MaxPool1d(4, stride=4)

        # Block 2 — mid-frequency features (spindles, K-complexes)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=15, stride=2, padding=7)
        self.bn2   = nn.BatchNorm1d(32)
        self.pool2 = nn.MaxPool1d(4, stride=4)

        # Block 3 — high-frequency features (beta, gamma)
        self.conv3 = nn.Conv1d(32, 64, kernel_size=7, stride=1, padding=3)
        self.bn3   = nn.BatchNorm1d(64)
        self.pool3 = nn.MaxPool1d(2, stride=2)

        # Global pooling + classifier head
        self.adaptive_pool = nn.AdaptiveAvgPool1d(16)
        self.fc1     = nn.Linear(64 * 16, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2     = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


# Sanity check
_dummy = torch.randn(64, 1, 7681)   # 30 s × 256 Hz + 1 sample
_model = SleepCNN(num_classes=5)
_out   = _model(_dummy)
print(f"Input  shape : {_dummy.shape}")
print(f"Output shape : {_out.shape}  ← expected [64, 5]")
del _dummy, _model, _out


## 8. Train the 5-Class Model
- **Loss:** `CrossEntropyLoss` with inverse-frequency class weights to counteract N2 dominance.  
- **Optimizer:** Adam (lr = 1e-3).  
- **Scheduler:** `ReduceLROnPlateau` — halves lr after 2 epochs without val-accuracy improvement.  
- **Checkpointing:** best model (by val accuracy) is saved to Drive.

> ⏱ ~2 hours on Colab T4 GPU.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device.type.upper()}")

# Inverse-frequency class weights
# Counts from EDA (Section 6) — update if your dataset differs.
raw_counts    = torch.tensor([13028., 2991., 51050., 5225., 13565.])
class_weights = (raw_counts.sum() / (5 * raw_counts)).to(device)
print("Class weights:", [f"{w:.3f}" for w in class_weights])

# Model, loss, optimizer, scheduler
model     = SleepCNN(num_classes=5).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2,
)

# Training configuration
STAGE_NAMES   = ["Wake", "N1", "N2", "N3", "REM"]
NUM_EPOCHS    = 15
CHECKPOINT    = "/content/drive/MyDrive/Sleep Classification Project/best_sleep_cnn_5class.pth"
best_val_acc  = 0.0
train_acc_history, val_acc_history = [], []

# Training loop
for epoch in range(1, NUM_EPOCHS + 1):

    # Train
    model.train()
    run_loss = correct = total = 0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()

        run_loss += loss.item()
        correct  += (logits.argmax(1) == y).sum().item()
        total    += y.size(0)

    train_acc = 100 * correct / total
    train_acc_history.append(train_acc)

    # Validate
    model.eval()
    val_correct = val_total = 0
    class_correct = [0] * 5
    class_total   = [0] * 5

    with torch.no_grad():
        for X, y in val_loader:
            X, y  = X.to(device), y.to(device)
            preds = model(X).argmax(1)
            val_correct += (preds == y).sum().item()
            val_total   += y.size(0)
            for c in range(5):
                mask = (y == c)
                class_correct[c] += (preds[mask] == c).sum().item()
                class_total[c]   += mask.sum().item()

    val_acc = 100 * val_correct / val_total
    val_acc_history.append(val_acc)
    scheduler.step(val_acc)

    print(f"\nEpoch {epoch:02d}/{NUM_EPOCHS}  —  "
          f"Train: {train_acc:.2f}%   Val: {val_acc:.2f}%")
    for c in range(5):
        ct  = class_total[c]
        pct = 100 * class_correct[c] / ct if ct > 0 else 0.0
        print(f"  {STAGE_NAMES[c]:5s}: {class_correct[c]:4d}/{ct:4d}  ({pct:.1f} %)")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT)
        print(f"  ★ New best model saved ({best_val_acc:.2f} %)")

print(f"\nTraining complete.  Best val accuracy: {best_val_acc:.2f} %")


## 9. Results — 5-Class Model
Visualize accuracy curves and evaluate the best checkpoint on the validation set.
Pre-recorded training history is included so the plots render even without re-running the training loop.


In [ ]:
# Pre-recorded training history (from our training run)
STAGE_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
NUM_EPOCHS  = 15

train_acc_history = [
    42.08, 51.27, 56.18, 59.78, 61.01,
    61.45, 64.80, 66.32, 68.30, 68.84,
    69.81, 69.68, 69.44, 70.13, 71.35,
]
val_acc_history = [
    47.08, 33.12, 47.46, 60.34, 51.68,
    52.87, 53.12, 58.17, 57.99, 65.92,
    62.55, 53.24, 66.82, 60.55, 55.35,
]
BEST_EPOCH     = 13
BEST_VAL_ACC   = 66.82

# Figure 1: Accuracy curves
fig, ax = plt.subplots(figsize=(8, 5))
epochs_range = range(1, NUM_EPOCHS + 1)
ax.plot(epochs_range, train_acc_history, color="steelblue",  linewidth=2, label="Train Accuracy")
ax.plot(epochs_range, val_acc_history,   color="darkorange", linewidth=2, label="Val Accuracy")
ax.axvline(x=BEST_EPOCH, color="green", linestyle="--", linewidth=1.5,
           label=f"Best Checkpoint (Epoch {BEST_EPOCH})")
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Accuracy (%)", fontsize=11)
ax.set_title("5-Class Model: Training vs Validation Accuracy", fontsize=13)
ax.set_xticks(epochs_range)
ax.legend(fontsize=11)
fig.tight_layout()
fig.savefig("/content/drive/MyDrive/Sleep Classification Project/accuracy_curve_5class.png", dpi=150)
plt.show()

# Figure 2: Per-class accuracy bar chart
correct_5 = [1978, 295, 4074, 638, 1478]
total_5   = [2538, 530, 7170, 737, 1691]
per_class = [100 * c / t for c, t in zip(correct_5, total_5)]

fig, ax = plt.subplots(figsize=(7, 4))
colors  = ["steelblue", "salmon", "mediumseagreen", "mediumpurple", "goldenrod"]
bars    = ax.bar(STAGE_NAMES, per_class, color=colors)
ax.set_ylim(0, 110)
ax.set_ylabel("Accuracy (%)", fontsize=11)
ax.set_title(f"Per-Class Accuracy at Best Epoch ({BEST_EPOCH}) — Val: {BEST_VAL_ACC} %", fontsize=11)
for bar, acc in zip(bars, per_class):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f"{acc:.1f} %", ha="center", fontsize=11, fontweight="bold")
fig.tight_layout()
fig.savefig("/content/drive/MyDrive/Sleep Classification Project/per_class_accuracy_5class.png", dpi=150)
plt.show()

# Figure 3: Confusion matrix (requires model + val_loader to be in scope)
try:
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in val_loader:
            preds = model(X.to(device)).argmax(1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(y.tolist())

    cm  = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=STAGE_NAMES).plot(
        ax=ax, cmap="Blues", colorbar=False,
    )
    ax.set_title(f"Confusion Matrix — Best Model (Epoch {BEST_EPOCH})", fontsize=13)
    fig.tight_layout()
    fig.savefig("/content/drive/MyDrive/Sleep Classification Project/confusion_matrix_5class.png", dpi=150)
    plt.show()

    print("\nClassification Report — 5-Class Model:")
    print(classification_report(all_labels, all_preds, target_names=STAGE_NAMES, digits=3))

except NameError:
    print("Skipping confusion matrix — run Section 8 first to populate `model` and `val_loader`.")

# Summary
print("─" * 50)
print(f"Best Epoch      : {BEST_EPOCH} / {NUM_EPOCHS}")
print(f"Best Val Acc    : {BEST_VAL_ACC} %")
print(f"Final Train Acc : {train_acc_history[-1]:.2f} %")
print("\nPer-class accuracy at best epoch:")
for name, c, t in zip(STAGE_NAMES, correct_5, total_5):
    print(f"  {name:5s}: {c}/{t}  ({100*c/t:.1f} %)")


## 10. 3-Class Model: Wake / NREM / REM
N1, N2, and N3 are merged into a single **NREM** class. The hypothesis is that reducing
inter-class confusion between similar NREM stages should yield higher overall accuracy —
and it does: the 3-class model reaches **82.2 % validation accuracy**.

The `SleepIterableDataset` defined in Section 4 accepts an optional `label_map` argument
that performs the remapping on-the-fly with zero additional preprocessing.


In [ ]:
# Label mapping
LABEL_MAP_3CLASS = {0: 0, 1: 1, 2: 1, 3: 1, 4: 2}   # NREM = N1 + N2 + N3
STAGE_NAMES_3    = ["Wake", "NREM", "REM"]

# DataLoaders
train_dataset_3 = SleepIterableDataset(train_x, train_y, shuffle=True,  label_map=LABEL_MAP_3CLASS)
val_dataset_3   = SleepIterableDataset(val_x,   val_y,   shuffle=False, label_map=LABEL_MAP_3CLASS)

train_loader_3 = DataLoader(train_dataset_3, batch_size=BATCH_SIZE)
val_loader_3   = DataLoader(val_dataset_3,   batch_size=BATCH_SIZE)

X_b3, y_b3 = next(iter(train_loader_3))
print(f"Batch X : {X_b3.shape}")
print(f"Labels  : {torch.unique(y_b3).tolist()}  ← expected [0, 1, 2]")

# Class weights
raw_counts_3    = torch.tensor([13028., 59266., 13565.])   # Wake / NREM / REM
class_weights_3 = (raw_counts_3.sum() / (3 * raw_counts_3)).to(device)
print(f"\nClass weights: {[f'{w:.3f}' for w in class_weights_3]}")

# Model, loss, optimizer
model_3     = SleepCNN(num_classes=3).to(device)
criterion_3 = nn.CrossEntropyLoss(weight=class_weights_3)
optimizer_3 = optim.Adam(model_3.parameters(), lr=1e-3)
scheduler_3 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_3, mode="max", factor=0.5, patience=2,
)

NUM_EPOCHS_3   = 15
CHECKPOINT_3   = "/content/drive/MyDrive/Sleep Classification Project/best_sleep_cnn_3class.pth"
best_val_acc_3 = 0.0
train_acc_history_3, val_acc_history_3 = [], []

# Training loop
for epoch in range(1, NUM_EPOCHS_3 + 1):

    model_3.train()
    run_loss = correct = total = 0

    for X, y in train_loader_3:
        X, y = X.to(device), y.to(device)
        optimizer_3.zero_grad()
        logits = model_3(X)
        loss   = criterion_3(logits, y)
        loss.backward()
        optimizer_3.step()

        run_loss += loss.item()
        correct  += (logits.argmax(1) == y).sum().item()
        total    += y.size(0)

    train_acc = 100 * correct / total
    train_acc_history_3.append(train_acc)

    model_3.eval()
    val_correct = val_total = 0
    class_correct_3 = [0] * 3
    class_total_3   = [0] * 3

    with torch.no_grad():
        for X, y in val_loader_3:
            X, y  = X.to(device), y.to(device)
            preds = model_3(X).argmax(1)
            val_correct += (preds == y).sum().item()
            val_total   += y.size(0)
            for c in range(3):
                mask = (y == c)
                class_correct_3[c] += (preds[mask] == c).sum().item()
                class_total_3[c]   += mask.sum().item()

    val_acc = 100 * val_correct / val_total
    val_acc_history_3.append(val_acc)
    scheduler_3.step(val_acc)

    print(f"\nEpoch {epoch:02d}/{NUM_EPOCHS_3}  —  "
          f"Train: {train_acc:.2f}%   Val: {val_acc:.2f}%")
    for c in range(3):
        ct  = class_total_3[c]
        pct = 100 * class_correct_3[c] / ct if ct > 0 else 0.0
        print(f"  {STAGE_NAMES_3[c]:5s}: {class_correct_3[c]:4d}/{ct:4d}  ({pct:.1f} %)")

    if val_acc > best_val_acc_3:
        best_val_acc_3 = val_acc
        torch.save(model_3.state_dict(), CHECKPOINT_3)
        print(f"  ★ New best model saved ({best_val_acc_3:.2f} %)")

print(f"\nDone.  Best 3-class val accuracy: {best_val_acc_3:.2f} %")

# Plots
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, NUM_EPOCHS_3 + 1), train_acc_history_3, color="steelblue",  linewidth=2, label="Train")
ax.plot(range(1, NUM_EPOCHS_3 + 1), val_acc_history_3,   color="darkorange", linewidth=2, label="Val")
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Accuracy (%)", fontsize=11)
ax.set_title("3-Class Model: Training vs Validation Accuracy", fontsize=13)
ax.set_xticks(range(1, NUM_EPOCHS_3 + 1))
ax.legend(fontsize=11)
fig.tight_layout()
fig.savefig("/content/drive/MyDrive/Sleep Classification Project/accuracy_curve_3class.png", dpi=150)
plt.show()

# Confusion matrix
all_preds_3, all_labels_3 = [], []
model_3.eval()
with torch.no_grad():
    for X, y in val_loader_3:
        preds = model_3(X.to(device)).argmax(1).cpu()
        all_preds_3.extend(preds.tolist())
        all_labels_3.extend(y.tolist())

cm_3 = confusion_matrix(all_labels_3, all_preds_3)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm_3, display_labels=STAGE_NAMES_3).plot(
    ax=ax, cmap="Blues", colorbar=False,
)
ax.set_title("Confusion Matrix — 3-Class Model", fontsize=13)
fig.tight_layout()
fig.savefig("/content/drive/MyDrive/Sleep Classification Project/confusion_matrix_3class.png", dpi=150)
plt.show()

print("\nClassification Report — 3-Class Model:")
print(classification_report(all_labels_3, all_preds_3, target_names=STAGE_NAMES_3, digits=3))
